In [ ]:
import pandas as pd

In [ ]:
def normalize_series(s):
    return (
        s.astype("string")  # pandas string dtype (handles NA properly)
         .str.lower()
         .fillna("")
    )

def competition_aware_merge(df_peak, df_new):
    result = df_peak.copy().astype("object")
    df_new = df_new.astype("object")

    meta_cols = [c for c in result.columns if c not in ['ID', 'PXD', 'Raw Data File']]

    for col in meta_cols:
        peak_norm = result[col].astype(str).str.lower().fillna("")
        new_norm  = df_new[col].astype(str).str.lower().fillna("")

        is_missing = peak_norm.isin(['not applicable', 'nan', 'none', ''])
        has_info   = ~new_norm.isin(['not applicable', 'nan', 'none', ''])

        mask = is_missing & has_info
        result.loc[mask, col] = df_new.loc[mask, col]

    return result

In [ ]:
from difflib import SequenceMatcher
def calculate_fill_stability(df_anchor, df_new):
    """
    Simulates the Agglomerative Clustering logic at a 0.8 threshold.
    Higher 'Stability' = Safe.
    Higher 'Fill' = Potential  growth.
    """
    stats = {"gains": 0, "corruptions": 0, "perfect": 0}
    meta_cols = [c for c in df_anchor.columns if c not in ['ID', 'PXD', 'Raw Data File']]
    
    for col in meta_cols:
        for a, n in zip(df_anchor[col].astype(str), df_new[col].astype(str)):
            a_clean, n_clean = a.strip().lower(), n.strip().lower()
            
            # Case 1: Filling a hole (The Goal)
            if a_clean in ['not applicable', 'nan'] and n_clean not in ['not applicable', 'nan']:
                stats["gains"] += 1
            
            # Case 2: Changing existing data (The Risk)
            elif a_clean not in ['not applicable', 'nan']:
                sim = SequenceMatcher(None, a_clean, n_clean).ratio()
                if sim >= 0.8:
                    stats["perfect"] += 1 # Stays in the same cluster
                else:
                    stats["corruptions"] += 1 # Forces a new cluster (BAD)
                    
    print(f"--- Metric Analysis ---")
    print(f"Recall Gains: {stats['gains']} holes filled.")
    print(f"Precision Risks: {stats['corruptions']} clusters broken.")
    return stats       

In [ ]:
from pathlib import Path
ALL_MODELS = [
    ("claude-sonnet-4-6", r"data\claude-sonnet-4-6\submission_claude_sonnet-4.6.R1.R2.csv", "closed", ["consensus","holes","submission4"],1),
    ("gemma-3-4b", r"data\hdd2026\submission_gemma-3-4b-it_T0_R3_normalized\submission.csv", "open", ["consensus","holes","submission4"],2),
    ("gpt-5.4", r"data\gpt-5.4\submission_gpt-5.4_R1.csv", "closed", ["consensus","holes"]),
    ("qwen3-4b", r"data\hdd2026\submission_qwen3-4b\submission_qwen3-4b_T0_R3_normalized.csv", "open", ["consensus","holes","submission4"],3),
    ("deepseek-coder-v2-lite-instruct", r"data\hdd2026\submission_deepseek-coder-v2-lite-instruct_T0_R3_normalized\submission.csv", "open", ["consensus", "holes","submission4"],4),
    ("medgemma-4b-it",r"data\hdd2026\submission_medgemma-4b-it_T0_R3_normalized\submission.csv", "open", ["consensus","holes"]),
    ("nanbeige4.1-3b-q8", r"data\hdd2026\submission_nanbeige4.1-3b-q8_T0_R2_normalized\submission.csv", "open", ["consensus","holes"]),
    ("LocalAI-functioncall-llama3.2-3b-v0.5",r"data\hdd2026\submission_LocalAI-functioncall-llama3.2-3b-v0.5_T0_R3_normalized\submission.csv", "open", ["consensus","holes"]),
    ("gpt-o4-mini",r"data\gpt-o4-mini\submission_gpt-o4-mini.csv","closed", ["consensus","holes"]),
    ("bggpt-gemma-3-27bgpt-o4-min-fp8", r"data\bggpt-gemma-3-27b-fp8\submission_bggpt-gemma-3-27b-fp8_R1.csv", "closed", ["consensus","holes"])
]

fill_holes_merge = {"open" : [] , "closed" : [], "all" : [], "submission4" : []}
consensus_merge = {"open" : {"names":[] , "inputs" : []}, "closed" :{"names":[] , "inputs" : []}, "all" : {"names":[] , "inputs" : []}}
for m in ALL_MODELS:
    print(m)
    assert Path(m[1]).exists()
    if "submission4" in m[3]:
        fill_holes_merge["submission4"].append(m[1])
    if "holes" in m[3]:
        fill_holes_merge[m[2]].append(m[1])
        fill_holes_merge["all"].append(m[1])
    if "consensus" in m[3]:
        consensus_merge[m[2]]["inputs"].append(m[1])
        consensus_merge[m[2]]["names"].append(m[0])
        consensus_merge["all"]["inputs"].append(m[1])
        consensus_merge["all"]["names"].append(m[0])


In [ ]:
fill_holes_merge

In [ ]:

def merge_holes(models, thismodel = None):
    merged = thismodel
    new_df = None
    import re 
    for model in models:
        print(model)
        df = pd.read_csv(model)
        df.replace(r"\|", ";", regex=True,  inplace=True)
        df.replace(';none', '', inplace=True)
        df.replace(';Not Applicable', '', inplace=True)
        pattern = re.compile(r"^none$", re.I)
        df = df.replace(pattern, "Not Applicable")     
        holes = (df.apply(lambda col: col.astype(str).str.lower() == "not applicable")).sum().sum()
        print(f"====== {model}\t holes {holes}")
        #display(df.head())
        if merged is not None:
            result = calculate_fill_stability(merged, df)
            print(result)
            merged = competition_aware_merge(merged, df)
        else:
            merged = df
        holes = (merged.apply(lambda col: col.astype(str).str.lower() == "not applicable")).sum().sum()   
        print(f"Holes {holes}") 
    return merged
        #    f1_merged, harm_norm, harm_subm, eval_df_merged = score(df, df_merged, row_id_column_name="ID")

#for tag in fill_holes_merge:
tag = "open"
print(f"==== {tag} ===")
merged = merge_holes(fill_holes_merge[tag])
merged.to_csv(f"submission.{tag}.holes.csv")

In [ ]:
import re
index = 1
df = pd.read_csv(ALL_MODELS[index][1])
print(ALL_MODELS[index][0])
df.replace(r"\|", ";", regex=True,  inplace=True)
df.replace(';none', '', inplace=True)
df.replace(';Not Applicable', '', inplace=True)
pattern = re.compile(r"^none$", re.I)
df = df.replace(pattern, "Not Applicable")     

merged = merge_holes(fill_holes_merge["all"],df)

    

In [ ]:
for tag in consensus_merge:
    print(f"rem consensus {tag} models")
    out = f"--out_sdrf submission.{tag}.csv --out_matrix comparison_matrix.{tag}.csv "
    print("uv run comparison.py ","--names"," ".join(consensus_merge[tag]["names"]),"--inputs", " ".join(consensus_merge[tag]["inputs"]), out)

In [ ]:
df = pd.DataFrame(ALL_MODELS)
df.columns = ["models","path","type","report"]
df.to_csv("models_compared.csv")